# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaaDasim05/Flyrank-ML-Internship/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [28]:
%pip -q install duckdb huggingface_hub


In [29]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [30]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [31]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [32]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            -- Prediction period: last 45 days
            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 45 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_last45,

            -- Historical period: previous 45 days
            SUM(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 45 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev45,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 45 DAY
                    THEN f.gsc_clicks
                    ELSE 0
                END
            ) AS clk_last45,

            AVG(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 45 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_prev45,

            -- NEW FEATURE: position volatility from the historical period
            STDDEV_SAMP(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 45 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_volatility_prev45

        FROM {TABLES['fact_daily']} f, bounds b

        WHERE f.report_date > b.end_d - INTERVAL 90 DAY

        GROUP BY 1, 2

        HAVING imp_prev45 >= 100
    )

    SELECT *
    FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

123,299 content items with enough history


,client_hash_id,content_hash_id,imp_last45,imp_prev45,clk_last45,pos_prev45,pos_volatility_prev45
0,client_62f4a7e64f5e0096,content_904afdcdd174c282,1590.0,4906.0,3.0,5.415263,2.554386
1,client_62f4a7e64f5e0096,content_f9fdb2068b96b2f7,20.0,127.0,0.0,34.277249,29.262473
2,client_62f4a7e64f5e0096,content_d0ad7ff3a79b6f9b,1341.0,685.0,3.0,2.339134,2.394992
3,client_62f4a7e64f5e0096,content_8cb258edd1d7494a,137.0,470.0,1.0,8.851886,6.286416
4,client_62f4a7e64f5e0096,content_6f12f6a057901255,241.0,133.0,1.0,7.430181,5.962834


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [33]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 123,299 rows


,client_hash_id,content_hash_id,imp_last45,imp_prev45,clk_last45,pos_prev45,pos_volatility_prev45,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_62f4a7e64f5e0096,content_904afdcdd174c282,1590.0,4906.0,3.0,5.415263,2.554386,38.0,0.077894,0.750616,146.0,1114.0,0.131059
1,client_62f4a7e64f5e0096,content_f9fdb2068b96b2f7,20.0,127.0,0.0,34.277249,29.262473,2.0,0.238095,0.598639,13.0,24.0,0.541667
2,client_62f4a7e64f5e0096,content_d0ad7ff3a79b6f9b,1341.0,685.0,3.0,2.339134,2.394992,7.0,0.169299,0.767522,39.0,128.0,0.304688
3,client_62f4a7e64f5e0096,content_8cb258edd1d7494a,137.0,470.0,1.0,8.851886,6.286416,4.0,0.072488,0.784185,49.0,87.0,0.563218
4,client_62f4a7e64f5e0096,content_6f12f6a057901255,241.0,133.0,1.0,7.430181,5.962834,1.0,0.050802,0.901070,18.0,18.0,1.000000


# intership code section

In [34]:
data.columns.tolist()

['client_hash_id',
 'content_hash_id',
 'imp_last45',
 'imp_prev45',
 'clk_last45',
 'pos_prev45',
 'pos_volatility_prev45',
 'visible_queries',
 'rare_share',
 'anon_share',
 'top_query_impressions',
 'kept_impressions',
 'top_query_share']

In [35]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import classification_report

# ---------------------------------------------------------
# Create the 45-day decline label
# ---------------------------------------------------------

data["is_declining_45"] = (
    data["imp_last45"] < 0.8 * data["imp_prev45"]
).astype(int)

# Features available before the prediction period
feature_cols = [
    "imp_prev45",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share",
    "pos_volatility_prev45"
]

model_data = data.dropna(
    subset=feature_cols + ["client_hash_id"]
).copy()

X = model_data[feature_cols]
y = model_data["is_declining_45"]
groups = model_data["client_hash_id"]

print(f"Usable rows: {len(model_data):,}")
print(f"Number of clients: {groups.nunique()}")
print(f"Decline rate: {y.mean():.3f}")

Usable rows: 108,044
Number of clients: 49
Decline rate: 0.629


In [36]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import classification_report

# =========================================================
# 1. RANDOM TRAIN / TEST SPLIT
# =========================================================

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

random_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

random_model.fit(X_tr, y_tr)

random_pred = random_model.predict(X_te)

print("=" * 60)
print("RANDOM SPLIT")
print("=" * 60)

print(classification_report(
    y_te,
    random_pred,
    digits=3
))


# =========================================================
# 2. CLIENT-LEVEL GROUP SPLIT
# =========================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_group_train = X.iloc[train_idx]
X_group_test = X.iloc[test_idx]

y_group_train = y.iloc[train_idx]
y_group_test = y.iloc[test_idx]

group_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

group_model.fit(
    X_group_train,
    y_group_train
)

group_pred = group_model.predict(X_group_test)

print("=" * 60)
print("CLIENT GROUP SPLIT")
print("=" * 60)

print(classification_report(
    y_group_test,
    group_pred,
    digits=3
))

print("Clients in training:", groups.iloc[train_idx].nunique())
print("Clients in testing:", groups.iloc[test_idx].nunique())

RANDOM SPLIT
              precision    recall  f1-score   support

           0      0.744     0.578     0.651     10010
           1      0.780     0.883     0.828     17001

    accuracy                          0.770     27011
   macro avg      0.762     0.730     0.740     27011
weighted avg      0.767     0.770     0.763     27011

CLIENT GROUP SPLIT
              precision    recall  f1-score   support

           0      0.765     0.630     0.691      4548
           1      0.839     0.908     0.872      9614

    accuracy                          0.819     14162
   macro avg      0.802     0.769     0.782     14162
weighted avg      0.815     0.819     0.814     14162

Clients in training: 36
Clients in testing: 13


In [37]:
import pandas as pd

importance = pd.Series(
    group_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("Feature importance:")
print(importance)

Feature importance:
imp_prev45               0.224204
anon_share               0.181506
rare_share               0.174657
pos_volatility_prev45    0.150088
top_query_share          0.136556
visible_queries          0.132988
dtype: float64


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [38]:
# =========================================================
# 5. A first honest model — 90-day experiment
# =========================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import classification_report

# Label: impressions declined by more than 20%
data["is_declining_45"] = (
    data["imp_last45"] < 0.8 * data["imp_prev45"]
).astype(int)

# Features available before the prediction period
feature_cols = [
    "imp_prev45",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share",
    "pos_volatility_prev45"
]

model_data = data.dropna(
    subset=feature_cols + ["client_hash_id"]
).copy()

X = model_data[feature_cols]
y = model_data["is_declining_45"]
groups = model_data["client_hash_id"]

print(f"Usable rows: {len(model_data):,}")
print(f"Number of clients: {groups.nunique()}")
print(f"Decline rate: {y.mean():.3f}")


# =========================================================
# Random split
# =========================================================

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

random_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

random_model.fit(X_tr, y_tr)

random_pred = random_model.predict(X_te)

print("\n" + "=" * 60)
print("RANDOM SPLIT")
print("=" * 60)

print(classification_report(
    y_te,
    random_pred,
    digits=3
))


# =========================================================
# Client-level GroupShuffleSplit
# =========================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_group_train = X.iloc[train_idx]
X_group_test = X.iloc[test_idx]

y_group_train = y.iloc[train_idx]
y_group_test = y.iloc[test_idx]

group_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

group_model.fit(
    X_group_train,
    y_group_train
)

group_pred = group_model.predict(X_group_test)

print("\n" + "=" * 60)
print("CLIENT GROUP SPLIT")
print("=" * 60)

print(classification_report(
    y_group_test,
    group_pred,
    digits=3
))

print("Clients in training:", groups.iloc[train_idx].nunique())
print("Clients in testing:", groups.iloc[test_idx].nunique())


# =========================================================
# Feature importance
# =========================================================

importance = pd.Series(
    group_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\nFeature importance:")
print(importance)

Usable rows: 108,044
Number of clients: 49
Decline rate: 0.629

RANDOM SPLIT
              precision    recall  f1-score   support

           0      0.744     0.578     0.651     10010
           1      0.780     0.883     0.828     17001

    accuracy                          0.770     27011
   macro avg      0.762     0.730     0.740     27011
weighted avg      0.767     0.770     0.763     27011


CLIENT GROUP SPLIT
              precision    recall  f1-score   support

           0      0.765     0.630     0.691      4548
           1      0.839     0.908     0.872      9614

    accuracy                          0.819     14162
   macro avg      0.802     0.769     0.782     14162
weighted avg      0.815     0.819     0.814     14162

Clients in training: 36
Clients in testing: 13

Feature importance:
imp_prev45               0.224204
anon_share               0.181506
rare_share               0.174657
pos_volatility_prev45    0.150088
top_query_share          0.136556
visible_que